In [10]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'

with h5py.File(filename, 'r') as f:
    #low_cat_matrix = f['data/lowest_tier_category_matrix2'][:]
    high_cat_matrix = f['data/highest_tier_category_matrix2'][:] # Done later
    arch_gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

arch_category = [x.decode('UTF-8') for x in category_names]
arch_gene_names = [x.decode('UTF-8') for x in arch_gene_names]

category_list  = ["Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]

category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

In [17]:
# Compute row-wise averages while skipping NaN values
row_averages = np.nanmean(high_cat_matrix, axis=1)

# Create new matrix where each value is divided by row average
high_cat_oe_matrix = np.divide(high_cat_matrix, row_averages[:, np.newaxis], 
                       out=np.zeros_like(high_cat_matrix), where=row_averages[:, np.newaxis]!=0)

high_cat_log2_oe_matrix = np.log2(high_cat_oe_matrix)

---
# Functions

In [11]:
def get_gene_indexes(gene_names):
    gene_indexes = []
    new_gene_names = []
    for gene in gene_names:
        try:
            index = arch_gene_names.index(gene)
            gene_indexes.append(index)
            new_gene_names.append(gene)
        except ValueError:
            pass
    return new_gene_names, gene_indexes

#gene_names, gene_indexes = get_gene_indexes(gene_names)

In [12]:
def get_geneset_category_means(matrix):
    
    # Calculate the column-wise average
    geneset_mean_row = np.mean(matrix, axis=0)

    return geneset_mean_row
    

---
# Main

In [18]:
# Initialize the final dictionary with empty lists for each category
folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
filename = folder + "h.c1.c2.all.v7.5.1.symbols.gmt"

geneset_means_matrix = []
geneset_names = []
gene_names = []

with open(filename, 'r', ) as file:
    
    for line in file:
#         print(line)
        
        # Extract the geneset name and the corresponding genes 
        line = line.strip().split('\t')
        geneset_names.append(line[0])
        gene_names = line[2:]
#         print(geneset_names, ":", gene_names)
        
        # Match geneset genes to the ARCHS4 gene names list 
        # If gene not found, remove from the gene_names
        # Otherwise, return a list of the genes idsx within ARCHS4
        gene_names, gene_indexes = get_gene_indexes(gene_names)
#         print(gene_names, "\n", gene_indexes)
        
        extracted_matrix = high_cat_log2_oe_matrix[gene_indexes, :] 
#         print(extracted_matrix)
#          print("Extracted Matrix Shape:", extracted_matrix.shape)
        
        geneset_mean_row = get_geneset_category_means(extracted_matrix)
        geneset_means_matrix.append(geneset_mean_row)
        #print("Geneset Matrix Shape:", len(geneset_means_matrix), len(geneset_means_matrix[0]))
#         print()
#         print(geneset_means_matrix)

# View pandas df
system_list = ["Cardiovascular System","Connective Tissue","Digestive System","Immune System",
"Integumentary System","Muscular System","Nervous System","Respiratory System",
"Urogenital System"]
df = pd.DataFrame(geneset_means_matrix, columns=system_list, index=geneset_names)
print(df.shape)
        
df.to_csv('top10genesets_highest_cat_matrix.csv', index=True)
#display(df)



/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/lwu/.conda/envs/example_env/lib/python3.8/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in true_divide
  ret = um.true_divide(


(6715, 9)


---
# TOP GENESETS

In [20]:
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    #print('Indexes for Column "{}":'.format(col))
    #print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
BRUNEAU_SEPTATION_ATRIAL                                  2.525519
REACTOME_PHYSIOLOGICAL_FACTORS                            1.418212
REACTOME_VEGF_LIGAND_RECEPTOR_INTERACTIONS                1.162551
BRUNEAU_SEPTATION_VENTRICULAR                             1.117213
PID_VEGF_VEGFR_PATHWAY                                    1.079683
WP_SARSCOV2_ALTERING_ANGIOGENESIS_VIA_NRP1                1.041412
CASTELLANO_HRAS_AND_NRAS_TARGETS_DN                       0.993464
SANA_TNF_SIGNALING_DN                                     0.891840
WP_ROBO4_AND_VEGF_SIGNALING_PATHWAYS_CROSSTALK            0.842640
REACTOME_YAP1_AND_WWTR1_TAZ_STIMULATED_GENE_EXPRESSION    0.824762
Name: Cardiovascular System, dtype: float64

Top 10 values in Column "Connective Tissue":
MYLLYKANGAS_AMPLIFICATION_HOT_SPOT_1                               1.746927
MARTINELLI_IMMATURE_NEUTROPHIL_UP                                  1.596763
REACTOME_ERYTHROCYTES_TAKE_UP_OXYGEN_AN

In [16]:
# Old - used matrix before oe calc
# loop through each column and get the top 10 highest values and their indexes
for col in df.columns:
    top_10 = df[col].nlargest(10)
    print('Top 10 values in Column "{}":'.format(col))
    print(top_10)
    print('Indexes for Column "{}":'.format(col))
    print(top_10.index)
    print()

Top 10 values in Column "Cardiovascular System":
KEGG_RIBOSOME                                                           3.033843
REACTOME_EUKARYOTIC_TRANSLATION_ELONGATION                              3.007763
WP_CYTOPLASMIC_RIBOSOMAL_PROTEINS                                       2.905718
REACTOME_MODULATION_BY_MTB_OF_HOST_IMMUNE_SYSTEM                        2.665770
REACTOME_RESPONSE_OF_EIF2AK4_GCN2_TO_AMINO_ACID_DEFICIENCY              2.662340
BILANGES_SERUM_AND_RAPAMYCIN_SENSITIVE_GENES                            2.630459
NGO_MALIGNANT_GLIOMA_1P_LOH                                             2.601219
REACTOME_SRP_DEPENDENT_COTRANSLATIONAL_PROTEIN_TARGETING_TO_MEMBRANE    2.586494
REACTOME_EUKARYOTIC_TRANSLATION_INITIATION                              2.532049
YANAGISAWA_LUNG_CANCER_RECURRENCE                                       2.489556
Name: Cardiovascular System, dtype: float64
Indexes for Column "Cardiovascular System":
Index(['KEGG_RIBOSOME', 'REACTOME_EUKARYOTIC_TRANSLAT